In [1]:
import sys
import os

root_dir = r"D:/BIN/Project/agentic_chatbot"
os.chdir(root_dir)
sys.path.append(root_dir)

In [2]:
from app.graph.chains.recommend_car_chain import consider_demand_car, render_table_from_list_dict, build_recommendation_info, parse_price_range, detect_demand, get_fields
from app.graph.state import ChatState
from app.intent.llm_intent import intent_and_product_detector
from app.intent.intent_router import dectect_intent
from app.embeddings.embedding_manager import PROD_DOCS
from app.embeddings.embedding_manager import PROD_IDX
from app.graph.nodes.recommend_car import recommend_car_node
from app.graph.chains.retrieve_products_chain import retrieve_products_docs
from app.LLMs.qwen3 import qwen3
from app.graph.chains.retrieve_policy_chain import retrieve_policy_docs, build_policy_context, detect_policy_intent, rerank_with_boost, filter_by_policy, generate_policy_answer
from app.retrievers.registry import get_policy_retriever
import json
from app.retrievers.vector_store import VectorStore
from app.embeddings.embedding_manager import EmbeddingManager
from app.retrievers.retriever import RAGRetriever
from app.retrievers.registry import init_retrievers

c:\Users\Asus\AppData\Local\Programs\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1849.64it/s]
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [3]:
embedding_manager = EmbeddingManager()
policy_store = VectorStore("policy")
product_store = VectorStore("product")

policy_retriever = RAGRetriever(policy_store, embedding_manager)
product_retriever = RAGRetriever(product_store, embedding_manager)
init_retrievers(policy_retriever=policy_retriever, product_retriever=product_retriever)

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9373.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384
Vector store initialized. Collection: policy
Existing documents in collection: 8
Vector store initialized. Collection: product
Existing documents in collection: 30


In [4]:
PROMPT_REP = """
    You are an assistant that answers user questions based only on the provided context.

    Context:
    These are related documents: {context}

    
    User message:
    {user_message}
    
    Instructions:
    - Use the information explicitly stated in the context.
    - Reply as natural as possible, be willing but don't say hello at first.
    Answer in Vietnamese.
    """

In [5]:
def llm_rep(state: ChatState):
    prompt = PROMPT_REP.format(
        user_message = state.user_message,
        context = state.policy_context
    )
    response = qwen3.chat_completion(
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=150
    )
    
    state.response = response["choices"][0]["message"]["content"]
    return state

In [6]:
def retrieve_policy(state: ChatState) -> ChatState:
    policy_retriver = get_policy_retriever()
    query = state.user_message

    policy_codes = detect_policy_intent(query)

    raw_docs = policy_retriver.retrieve(query, top_k=10, score_threshold=0.0)

    if not raw_docs:
        state.retrieved_docs = []
        return state
    
    filtered_docs = filter_by_policy(raw_docs, policy_codes)

    reranked_docs = rerank_with_boost(filtered_docs, query)

    final_docs = reranked_docs[:3]

    state.retrieved_docs = final_docs
    return state

In [7]:
s1 = ChatState("tôi muốn hỏi về chính sách đổi trả xe của bên công ty")

In [8]:
s2 = retrieve_policy_docs(s1)
print(s2)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.52it/s]

Generated embeddings with shape: (1, 384)
ChatState(user_message='tôi muốn hỏi về chính sách đổi trả xe của bên công ty', intent=None, history=[], selected_car=None, compared_car=[], retrieved_docs=[{'id': 'doc_dae97a36_5', 'content': 'POL-06: Điều khoản đổi xe. Khách hàng được quyền đề nghị đổi xe trong vòng 07 ngày kể từ ngày nhận xe, với điều kiện xe chưa đăng ký biển số, chưa vận hành quá 300 km và không có dấu hiệu hư hỏng, tai nạn hoặc trầy xước do tác động bên ngoài. Việc đổi xe chỉ được thực hiện sau khi cơ sở tiến hành kiểm tra và xác nhận tình trạng xe đạt yêu cầu. Khách hàng phải chịu phí đổi xe tương đương 5% giá trị xe để bù đắp các chi phí liên quan.', 'metadata': {'doc_index': 5, 'code': 'POL-06:', 'source': 'D:\\BIN\\Project\\agentic_chatbot\\data\\raw\\customer_policy.txt', 'content_length': 436}, 'distance': 0.8386253118515015, 'score': 1.0213746881484986, 'rank': 4}], policy_context='', response='')


In [9]:
s3 = build_policy_context(s2)
print(s3)

ChatState(user_message='tôi muốn hỏi về chính sách đổi trả xe của bên công ty', intent=None, history=[], selected_car=None, compared_car=[], retrieved_docs=[{'id': 'doc_dae97a36_5', 'content': 'POL-06: Điều khoản đổi xe. Khách hàng được quyền đề nghị đổi xe trong vòng 07 ngày kể từ ngày nhận xe, với điều kiện xe chưa đăng ký biển số, chưa vận hành quá 300 km và không có dấu hiệu hư hỏng, tai nạn hoặc trầy xước do tác động bên ngoài. Việc đổi xe chỉ được thực hiện sau khi cơ sở tiến hành kiểm tra và xác nhận tình trạng xe đạt yêu cầu. Khách hàng phải chịu phí đổi xe tương đương 5% giá trị xe để bù đắp các chi phí liên quan.', 'metadata': {'doc_index': 5, 'code': 'POL-06:', 'source': 'D:\\BIN\\Project\\agentic_chatbot\\data\\raw\\customer_policy.txt', 'content_length': 436}, 'distance': 0.8386253118515015, 'score': 1.0213746881484986, 'rank': 4}], policy_context='POL-06: Điều khoản đổi xe. Khách hàng được quyền đề nghị đổi xe trong vòng 07 ngày kể từ ngày nhận xe, với điều kiện xe chưa đ

In [10]:
s4 = generate_policy_answer(s3)
print(s4)
print(s4.response)

ChatState(user_message='tôi muốn hỏi về chính sách đổi trả xe của bên công ty', intent=None, history=[], selected_car=None, compared_car=[], retrieved_docs=[{'id': 'doc_dae97a36_5', 'content': 'POL-06: Điều khoản đổi xe. Khách hàng được quyền đề nghị đổi xe trong vòng 07 ngày kể từ ngày nhận xe, với điều kiện xe chưa đăng ký biển số, chưa vận hành quá 300 km và không có dấu hiệu hư hỏng, tai nạn hoặc trầy xước do tác động bên ngoài. Việc đổi xe chỉ được thực hiện sau khi cơ sở tiến hành kiểm tra và xác nhận tình trạng xe đạt yêu cầu. Khách hàng phải chịu phí đổi xe tương đương 5% giá trị xe để bù đắp các chi phí liên quan.', 'metadata': {'doc_index': 5, 'code': 'POL-06:', 'source': 'D:\\BIN\\Project\\agentic_chatbot\\data\\raw\\customer_policy.txt', 'content_length': 436}, 'distance': 0.8386253118515015, 'score': 1.0213746881484986, 'rank': 4}], policy_context='POL-06: Điều khoản đổi xe. Khách hàng được quyền đề nghị đổi xe trong vòng 07 ngày kể từ ngày nhận xe, với điều kiện xe chưa đ

In [11]:
print(generate_policy_answer(s1).response)

Chính sách đổi xe của công ty quy định như sau: khách hàng có quyền đề nghị đổi xe trong vòng 07 ngày kể từ ngày nhận xe, với các điều kiện:

- Xe chưa đăng ký biển số.  
- Xe chưa vận hành quá 300 km.  
- Xe không có dấu hiệu hư hỏng, tai nạn hoặc trầy xước do tác động bên ngoài.  

Việc đổi xe chỉ được thực hiện sau khi cơ sở kiểm tra và xác nhận xe đạt yêu cầu. Khách hàng phải chịu phí đổi xe tương đương 5% giá trị xe để bù đắp chi phí liên quan.
